In [4]:
import os
import sys
import pandas as pd

# Permette di trovare config.py facendo un passo indietro nella cartella principale
sys.path.append(os.path.abspath(os.path.join('..')))

from config import PATH_CLEAN_RECIPES, PATH_CLEAN_INTERACTIONS
from models.collaborative_filtering import CollaborativeFilteringRecommender

# CARICAMENTO DEL DATASET (Questo crea la variabile df_recipes)
df_recipes = pd.read_csv(os.path.join('..', PATH_CLEAN_RECIPES))
df_interactions = pd.read_csv(os.path.join('..', PATH_CLEAN_INTERACTIONS))

print(f"Ricette caricate: {len(df_recipes)}")
print(f"Interazioni caricate: {len(df_interactions)}")

Ricette caricate: 15144
Interazioni caricate: 385801


In [ ]:
from models.collaborative_filtering import CollaborativeFilteringRecommender
from models.popularity import PopularityRecommender

# 1. Inizializziamo ed eseguiamo prima il modello di popolarità che useremo come paracadute per il cold start
pop_model = PopularityRecommender(m=50)
pop_model.fit(df_recipes, df_interactions)

# 2. Inizializziamo il Filtro Collaborativo impostando il limite minimo a 5 interazioni
cf_model = CollaborativeFilteringRecommender(min_user_interactions=5)
cf_model.fit(df_recipes, df_interactions)

# 3. TEST SCIENTIFICO: Comparamo i 4 algoritmi di Surprise sul nostro test set temporale
benchmark_results = cf_model.evaluate_algorithms()


# 4. TEST APPLICATIVO: Predizione per un utente reale registrato (es. prendiamo un ID che sappiamo essere attivo)
# Cerchiamo un utente attivo nel df delle interazioni per fare il test reale
utente_attivo = df_interactions['user_id'].value_counts().index[0]

print(f"\n" + "="*60 + "\n")
print(f"--- RACCOMANDAZIONE PERSONALIZZATA SVD PER UTENTE REALE: {utente_attivo} ---")
raccomandazioni_personali = cf_model.recommend(user_id=utente_attivo, top_k=3)

for r in raccomandazioni_personali:
    print(f"Ricetta consigliata: {r['name'].upper()} | Rating Predetto: {r['rating_predetto']} ⭐ | {r['calorie']} kcal")


# 5. TEST COLD START: Proviamo a passare un ID utente inventato di sana pianta (es. 99999999)
print(f"\n--- RACCOMANDAZIONE PER UTENTE IN COLD START (ID: 99999999) ---")
raccomandazioni_cold_start = cf_model.recommend(user_id=99999999, top_k=3, popularity_fallback_model=pop_model)

for r in raccomandazioni_cold_start:
    print(f"Ricetta (Fallback): {r['name'].upper()} | Score Popolarità: {r['score']} | {r['calorie']} kcal")

-> Filtro Cold Start applicato. Righe rimanenti: 365262
-> Split Temporale completato. Train size: 351077, Test size: 14185
-> Addestramento dell'algoritmo SVD definitivo...
-> Modello SVD addestrato con successo!

=== AVVIO BENCHMARK ALGORITMI (SURPRISE) ===
SVD (Fattori Latenti) -> RMSE: 0.6403 | MAE: 0.3835
